# grahspj: NGC 4395 and POX 52 From the VizieR SED Tool

This notebook follows the same pattern as `02_vizier_fairall9.ipynb`, but fits two low-mass AGN host galaxies in the same workflow: `NGC 4395` and `POX 52`.

Workflow:
- query broadband photometry for both targets from the CDS VizieR SED service
- keep only filters with transmission curves available to `grahspj`
- build one manual `FitConfig` per source
- run one fit per source
- inspect the fitted summaries and SED plots


## Prerequisites

This notebook assumes:
- you are running from the `notebooks/` directory of this repository
- `grahspj` dependencies are installed
- internet access is available for the VizieR SED query
- a valid DSPS SSP file is available

By default the notebook looks for `../jaxqsofit/tempdata.h5` as a sibling checkout. Adjust that path if your SSP file lives elsewhere.

Notes:
- the redshifts are fixed to the source spectroscopic redshifts; this notebook does not fit redshift or use a photo-z prior
- when multiple measurements map onto the same filter, the query helper keeps the entry with the smallest fractional flux uncertainty
- `fit_method="optax"` and the MAP stage of `fit_method="optax+nuts"` use staged MAP/SVI initialization by default


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

project_root = Path.cwd().resolve().parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from grahspj.config import AGNConfig, FilterSet, FitConfig, GalaxyConfig, InferenceConfig, LikelihoodConfig, Observation, PhotometryData
from grahspj.core import GRAHSPJ
from grahspj.mplstyle import style_path
from utils import query_vizier_sed

plt.style.use(style_path())


In [ ]:
targets = [
    {
        "name": "NGC 4395",
        "redshift": 0.001064,
        "search_radius_arcsec": 5.0,
        "prior_config": {
            "log_stellar_mass": {"loc": 8.5, "scale": 1.0},
            "fracAGN_5100": {"loc": 0.5, "scale": 0.25},
            "ebv_gal": {"scale": 0.15},
            "ebv_agn": {"scale": 0.15},
        },
    },
    {
        "name": "POX 52",
        "redshift": 0.0218,
        "search_radius_arcsec": 5.0,
        "prior_config": {
            "log_stellar_mass": {"loc": 9.0, "scale": 1.0},
            "fracAGN_5100": {"loc": 0.6, "scale": 0.25},
            "ebv_gal": {"scale": 0.15},
            "ebv_agn": {"scale": 0.15},
        },
    },
]


In [ ]:
target_results = {}

for target in targets:
    phot_rows, sed_table, vizier_url = query_vizier_sed(
        target["name"],
        radius=target["search_radius_arcsec"],
        host=("vizier.cfa.harvard.edu", 443, True),
        fallback_hosts=[("vizier.u-strasbg.fr", 80, False)],
        timeout=120,
    )
    target_results[target["name"]] = {
        "target": target,
        "phot_rows": phot_rows,
        "phot_table": Table(rows=phot_rows),
        "sed_table": sed_table,
        "vizier_url": vizier_url,
    }

    print(f"{target['name']}: {len(sed_table)} raw rows, {len(phot_rows)} supported points")
    print("Queried URL:", vizier_url)
    print("Filters:", [row["grahsp_filter"] for row in phot_rows])
    print()


In [ ]:
for name, result in target_results.items():
    print(name)
    display(result["phot_table"])


In [ ]:
dsps_ssp_fn = project_root.parent / "jaxqsofit" / "tempdata.h5"
assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"

def build_fit_config(target, phot_rows):
    return FitConfig(
        observation=Observation(
            object_id=target["name"].replace(" ", "_"),
            redshift=target["redshift"],
            fit_redshift=False,
        ),
        photometry=PhotometryData(
            filter_names=[row["grahsp_filter"] for row in phot_rows],
            fluxes=[row["flux_mjy"] for row in phot_rows],
            errors=[row["err_mjy"] for row in phot_rows],
            is_upper_limit=[False] * len(phot_rows),
            psf_fwhm_arcsec=[row["psf_fwhm_arcsec"] for row in phot_rows],
        ),
        filters=FilterSet(
            speclite_names={row["grahsp_filter"]: row["speclite_name"] for row in phot_rows},
            use_grahsp_database=False,
        ),
        galaxy=GalaxyConfig(dsps_ssp_fn=str(dsps_ssp_fn)),
        agn=AGNConfig(agn_type=1),
        likelihood=LikelihoodConfig(use_host_capture_model=True),
        inference=InferenceConfig(map_steps=400, learning_rate=5e-3),
        prior_config=target["prior_config"],
    )

configs = {
    name: build_fit_config(result["target"], result["phot_rows"])
    for name, result in target_results.items()
}

for name, cfg in configs.items():
    print(name)
    print("Configured filters:", cfg.photometry.filter_names)
    print("Configured redshift:", cfg.observation.redshift)
    print()


### Host-Capture Model

As in the Fairall 9 notebook, `use_host_capture_model=True` lets the host contribution vary with the effective spatial scale of each measurement. This is useful here because the VizieR SED points combine heterogeneous surveys with different angular resolution.

The AGN component is treated as unresolved, while the observed model flux is

$$F_{\mathrm{model},i} = F_{\mathrm{AGN},i} + \eta_i F_{\mathrm{host},i},$$

where $\eta_i$ is the fitted band-dependent host-capture fraction.


In [ ]:
fitters = {}
fit_results = {}

for name, cfg in configs.items():
    print(f"Fitting {name}")
    fitter = GRAHSPJ(cfg)
    fit_result = fitter.fit(
        fit_method="optax+nuts",
        prior_config=cfg.prior_config,
        dsps_ssp_fn=cfg.galaxy.dsps_ssp_fn,
        optax_steps=cfg.inference.map_steps,
        optax_lr=cfg.inference.learning_rate,
        nuts_warmup=100,
        nuts_samples=50,
        plot_fig=False,
        save_fig=False,
        save_result=False,
        progress_bar=True,
    )
    fitters[name] = fitter
    fit_results[name] = fit_result


In [ ]:
predictions = {}

for name, fitter in fitters.items():
    print(name)
    predictions[name] = fitter.predict()
    fig = fitter.plot_sed(show=True)


In [ ]:
for name, fitter in fitters.items():
    cfg = configs[name]
    pred = predictions[name]
    model_flux = np.asarray(pred["pred_fluxes"][0], dtype=float)
    host_flux = np.asarray(pred["host_fluxes"][0], dtype=float)
    dust_flux = np.asarray(pred["dust_fluxes"][0], dtype=float)
    agn_flux = np.asarray(pred["agn_fluxes"][0], dtype=float)
    phot_wave = np.asarray([flt.effective_wavelength for flt in fitter.context.filters], dtype=float)

    print(name)
    print("filter, eff_wave_A, obs_mJy, err_mJy, model_mJy, host_mJy, dust_mJy, agn_mJy")
    for row in zip(
        cfg.photometry.filter_names,
        phot_wave,
        cfg.photometry.fluxes,
        cfg.photometry.errors,
        model_flux,
        host_flux,
        dust_flux,
        agn_flux,
    ):
        print(row)
    print()


## Notes

- For a quick smoke test, change the fit loop to `fit_method="optax"` and skip the NUTS arguments.
- For fuller posterior exploration, keep `fit_method="optax+nuts"` and increase `nuts_warmup` and `nuts_samples`.
- If VizieR returns too few supported bands for either target, increase `search_radius_arcsec` cautiously or inspect `sed_table` to decide whether a manual photometry table is more appropriate.
- This notebook intentionally keeps `fit_redshift=False` for both sources.
